# GraphRAG Phase 8.1: POIナレッジグラフ構築

**目的**: 渋谷エリアのPOIデータからナレッジグラフを構築し、グラフRAG実験の基盤を作成する

**実行環境**: Google Colab (T4 GPU)

**作成日**: 2026-01-29

## 1. 環境セットアップ

In [ ]:
# Google Driveマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# プロジェクトディレクトリの設定
import os
import sys

# プロジェクトルート（Google Drive上のパスに変更してください）
PROJECT_ROOT = "/content/drive/MyDrive/experiments-local-llm"

# または、GitHubからクローン
if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/mopinfish/experiments-local-llm.git /content/experiments-local-llm
    PROJECT_ROOT = "/content/experiments-local-llm"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Current directory: {os.getcwd()}")

In [ ]:
# 依存ライブラリのインストール
!pip install -q networkx matplotlib pyvis pandas japanize_matplotlib

In [ ]:
# ライブラリのインポート
import json
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
from typing import Dict, List, Any, Tuple

# 日本語フォント対応
import japanize_matplotlib

# プロジェクトモジュール
from graph_builder import (
    POIGraphBuilder,
    get_pois_by_category,
    get_pois_by_area,
    get_nearby_pois,
    get_pois_in_same_area,
    find_pois_with_both_categories
)

print("Libraries imported successfully!")

## 2. POIデータの読み込みと確認

In [ ]:
# POIデータ読み込み
poi_file = os.path.join(PROJECT_ROOT, "poi_documents.json")

with open(poi_file, "r", encoding="utf-8") as f:
    pois = json.load(f)

print(f"Loaded {len(pois)} POIs")
print(f"\nSample POI:")
print(json.dumps(pois[0], ensure_ascii=False, indent=2))

In [ ]:
# カテゴリ分布の確認
category_counts = defaultdict(int)
for poi in pois:
    category = poi.get("metadata", {}).get("category", "不明")
    category_counts[category] += 1

# DataFrameで表示
df_categories = pd.DataFrame(
    [(cat, count) for cat, count in sorted(category_counts.items(), key=lambda x: -x[1])],
    columns=["Category", "Count"]
)
df_categories["Percentage"] = (df_categories["Count"] / len(pois) * 100).round(1)

print("Category Distribution:")
display(df_categories.head(20))

## 3. POIナレッジグラフの構築

In [ ]:
# グラフビルダーの初期化
builder = POIGraphBuilder(
    near_distance_threshold=100,  # 100m以内をNEAR_TOとする
    max_near_neighbors=20          # 各POIあたり最大20件のNEAR_TOエッジ
)

# グラフ構築
graph = builder.build_graph(
    pois,
    include_near_edges=True,
    include_same_area_edges=True,
    verbose=True
)

In [ ]:
# グラフ統計情報の確認
stats = builder.get_stats()

print("=" * 50)
print("Graph Statistics")
print("=" * 50)

print("\n[Nodes]")
for node_type, count in stats["nodes"].items():
    print(f"  {node_type:15}: {count:,}")

print("\n[Edges]")
for edge_type, count in stats["edges"].items():
    print(f"  {edge_type:15}: {count:,}")

## 4. グラフ構造の確認

In [ ]:
# ノードタイプ別の確認
node_types = defaultdict(list)
for node, data in graph.nodes(data=True):
    node_type = data.get("node_type", "unknown")
    node_types[node_type].append(node)

print("Node Types:")
for node_type, nodes in node_types.items():
    print(f"  {node_type}: {len(nodes)} nodes")
    if len(nodes) <= 5:
        for node in nodes:
            print(f"    - {node}")

In [ ]:
# エッジタイプ別の確認
edge_types = defaultdict(int)
for u, v, data in graph.edges(data=True):
    edge_type = data.get("edge_type", "unknown")
    edge_types[edge_type] += 1

print("Edge Types:")
for edge_type, count in sorted(edge_types.items(), key=lambda x: -x[1]):
    print(f"  {edge_type}: {count:,} edges")

In [ ]:
# カテゴリノードの確認
print("Category Nodes:")
for node in node_types.get("category", []):
    data = graph.nodes[node]
    print(f"  {data['name']}: {data['poi_count']} POIs")

In [ ]:
# エリアノードの確認
print("Area Nodes:")
area_data = []
for node in node_types.get("area", []):
    data = graph.nodes[node]
    area_data.append({
        "Area": data["name"],
        "Direction": data["direction"],
        "Zone": data["distance_zone"],
        "POI Count": data["poi_count"]
    })

df_areas = pd.DataFrame(area_data)
df_areas = df_areas.sort_values(["Zone", "Direction"]).reset_index(drop=True)
display(df_areas)

## 5. グラフクエリのテスト

In [ ]:
# カテゴリ別POI取得テスト
print("=" * 50)
print("Test: get_pois_by_category")
print("=" * 50)

for category in ["飲食店", "商店", "交通"]:
    pois_in_cat = get_pois_by_category(graph, category)
    print(f"\n{category}: {len(pois_in_cat)} POIs")
    
    # サンプル表示
    for poi_id in pois_in_cat[:3]:
        name = graph.nodes[poi_id].get("name", "不明")
        print(f"  - {name}")

In [ ]:
# エリア別POI取得テスト
print("=" * 50)
print("Test: get_pois_by_area")
print("=" * 50)

for area in ["east_near", "west_near", "north_mid"]:
    pois_in_area = get_pois_by_area(graph, area)
    print(f"\n{area}: {len(pois_in_area)} POIs")
    
    # サンプル表示
    for poi_id in pois_in_area[:3]:
        name = graph.nodes[poi_id].get("name", "不明")
        cat = graph.nodes[poi_id].get("subcategory", "不明")
        print(f"  - {name} ({cat})")

In [ ]:
# 近隣POI取得テスト
print("=" * 50)
print("Test: get_nearby_pois")
print("=" * 50)

# ランダムなPOIを選択
sample_poi = list(node_types["poi"])[0]
sample_name = graph.nodes[sample_poi].get("name", "不明")

print(f"\nBase POI: {sample_name}")
print(f"Node ID: {sample_poi}")

nearby = get_nearby_pois(graph, sample_poi, max_distance=100)
print(f"\nNearby POIs (within 100m): {len(nearby)}")

for poi_id, distance in nearby[:10]:
    name = graph.nodes[poi_id].get("name", "不明")
    cat = graph.nodes[poi_id].get("subcategory", "不明")
    print(f"  - {name} ({cat}) - {distance:.1f}m")

In [ ]:
# 2カテゴリ共存エリア検索テスト
print("=" * 50)
print("Test: find_pois_with_both_categories")
print("=" * 50)

# カフェと銀行が同じエリアにある場所を探す
results = find_pois_with_both_categories(graph, "飲食店", "金融", same_area=True)

print(f"\n飲食店 + 金融 が同じエリアにある場所: {len(results)} 件")

# エリアごとにグループ化して表示
by_area = defaultdict(list)
for r in results:
    by_area[r["area"]].append(r)

for area, items in list(by_area.items())[:5]:
    print(f"\n{area}:")
    for item in items[:3]:
        print(f"  - {item['poi1_name']} (飲食店) + {item['poi2_name']} (金融)")

## 6. グラフの可視化

In [ ]:
# カテゴリ・エリア構造のサブグラフを可視化
def create_structure_subgraph(graph: nx.DiGraph) -> nx.DiGraph:
    """カテゴリとエリアの構造のみを抽出したサブグラフを作成"""
    subgraph = nx.DiGraph()
    
    # ランドマーク、カテゴリ、エリアノードを追加
    for node, data in graph.nodes(data=True):
        if data.get("node_type") in ["landmark", "category", "area"]:
            subgraph.add_node(node, **data)
    
    # ADJACENT_TOエッジを追加
    for u, v, data in graph.edges(data=True):
        if data.get("edge_type") == "ADJACENT_TO":
            subgraph.add_edge(u, v, **data)
    
    return subgraph

structure_graph = create_structure_subgraph(graph)
print(f"Structure subgraph: {structure_graph.number_of_nodes()} nodes, {structure_graph.number_of_edges()} edges")

In [ ]:
# エリア構造の可視化
plt.figure(figsize=(14, 10))

# エリアノードのみ抽出
area_subgraph = nx.DiGraph()
for node, data in graph.nodes(data=True):
    if data.get("node_type") == "area":
        area_subgraph.add_node(node, **data)

for u, v, data in graph.edges(data=True):
    if data.get("edge_type") == "ADJACENT_TO":
        area_subgraph.add_edge(u, v, **data)

# ノード位置を方向と距離ゾーンに基づいて配置
pos = {}
direction_angles = {
    "north": 90, "northeast": 45, "east": 0, "southeast": -45,
    "south": -90, "southwest": -135, "west": 180, "northwest": 135,
    "center": 0
}
zone_radius = {"station": 0.5, "near": 1.5, "mid": 2.5, "far": 3.5}

import math
for node, data in area_subgraph.nodes(data=True):
    direction = data.get("direction", "center")
    zone = data.get("distance_zone", "mid")
    
    angle_deg = direction_angles.get(direction, 0)
    radius = zone_radius.get(zone, 2)
    
    angle_rad = math.radians(angle_deg)
    x = radius * math.cos(angle_rad)
    y = radius * math.sin(angle_rad)
    pos[node] = (x, y)

# ノードサイズをPOI数に応じて設定
node_sizes = [area_subgraph.nodes[n].get("poi_count", 10) * 50 for n in area_subgraph.nodes()]

# 距離ゾーンで色分け
zone_colors = {"station": "red", "near": "orange", "mid": "yellow", "far": "lightgreen"}
node_colors = [zone_colors.get(area_subgraph.nodes[n].get("distance_zone", "mid"), "gray") 
               for n in area_subgraph.nodes()]

# 描画
nx.draw_networkx_nodes(area_subgraph, pos, node_size=node_sizes, node_color=node_colors, alpha=0.7)
nx.draw_networkx_edges(area_subgraph, pos, edge_color="gray", alpha=0.5, arrows=True)
nx.draw_networkx_labels(area_subgraph, pos, 
                        labels={n: area_subgraph.nodes[n].get("name", n).replace("area:", "") 
                                for n in area_subgraph.nodes()},
                        font_size=8)

# 凡例
for zone, color in zone_colors.items():
    plt.scatter([], [], c=color, label=zone, s=100)
plt.legend(title="Distance Zone", loc="upper left")

# 渋谷駅を中心に表示
plt.scatter([0], [0], c="black", s=200, marker="*", zorder=5, label="渋谷駅")

plt.title("Area Structure Graph (Shibuya Station at Center)")
plt.axis("equal")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results/area_structure_graph.png"), dpi=150)
plt.show()

In [ ]:
# カテゴリ別POI分布の可視化
plt.figure(figsize=(12, 8))

# カテゴリ別のPOI数を集計
category_poi_counts = {}
for node, data in graph.nodes(data=True):
    if data.get("node_type") == "category":
        name = data.get("name", "不明")
        count = data.get("poi_count", 0)
        category_poi_counts[name] = count

# ソートして上位10カテゴリを表示
sorted_categories = sorted(category_poi_counts.items(), key=lambda x: -x[1])[:10]
categories = [c[0] for c in sorted_categories]
counts = [c[1] for c in sorted_categories]

plt.barh(categories[::-1], counts[::-1], color="steelblue")
plt.xlabel("Number of POIs")
plt.ylabel("Category")
plt.title("Top 10 Categories by POI Count")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results/category_distribution.png"), dpi=150)
plt.show()

## 7. インタラクティブ可視化（PyVis）

In [ ]:
# PyVisによるインタラクティブ可視化
from pyvis.network import Network

def visualize_subgraph_interactive(graph: nx.DiGraph, 
                                   center_poi: str = None,
                                   hop: int = 1,
                                   output_file: str = "graph.html"):
    """
    指定したPOIを中心としたサブグラフをインタラクティブに可視化
    
    Args:
        graph: 元のグラフ
        center_poi: 中心となるPOIノードID
        hop: 中心からのホップ数
        output_file: 出力HTMLファイル名
    """
    # サブグラフの抽出
    if center_poi:
        # 指定ホップ以内のノードを取得
        nodes = {center_poi}
        frontier = {center_poi}
        
        for _ in range(hop):
            next_frontier = set()
            for node in frontier:
                # 出ていくエッジ
                for _, target in graph.out_edges(node):
                    next_frontier.add(target)
                # 入ってくるエッジ
                for source, _ in graph.in_edges(node):
                    next_frontier.add(source)
            nodes.update(next_frontier)
            frontier = next_frontier
        
        subgraph = graph.subgraph(nodes)
    else:
        # カテゴリ・エリア・ランドマークのみ
        nodes = [n for n, d in graph.nodes(data=True) 
                 if d.get("node_type") in ["category", "area", "landmark"]]
        subgraph = graph.subgraph(nodes)
    
    # PyVisネットワーク作成（cdn_resources='in_line'でJupyter表示問題を回避）
    net = Network(height="600px", width="100%", notebook=True, directed=True, cdn_resources='in_line')
    
    # ノードタイプ別の色
    type_colors = {
        "poi": "#4CAF50",
        "category": "#2196F3",
        "subcategory": "#03A9F4",
        "area": "#FF9800",
        "landmark": "#F44336"
    }
    
    # ノード追加
    for node, data in subgraph.nodes(data=True):
        node_type = data.get("node_type", "unknown")
        label = data.get("name", node)
        color = type_colors.get(node_type, "#9E9E9E")
        
        # ノードサイズ
        if node_type == "landmark":
            size = 30
        elif node_type == "category":
            size = 25
        elif node_type == "area":
            size = 20
        else:
            size = 15
        
        net.add_node(node, label=label, color=color, size=size, title=str(data))
    
    # エッジ追加
    for u, v, data in subgraph.edges(data=True):
        edge_type = data.get("edge_type", "")
        net.add_edge(u, v, title=edge_type, label=edge_type[:10])
    
    # 物理演算設定
    net.toggle_physics(True)
    net.show_buttons(filter_=['physics'])
    
    # 保存
    output_path = os.path.join(PROJECT_ROOT, "results", output_file)
    net.save_graph(output_path)
    print(f"Interactive graph saved to: {output_path}")
    
    return net

In [ ]:
# 構造グラフ（カテゴリ・エリア）の可視化
net = visualize_subgraph_interactive(graph, center_poi=None, output_file="structure_graph.html")
net.show("structure_graph.html")

In [ ]:
# 特定POIを中心としたサブグラフの可視化
# 渋谷駅を中心に1ホップの範囲を表示
sample_poi = list(node_types["poi"])[0]
net = visualize_subgraph_interactive(graph, center_poi=sample_poi, hop=2, output_file="poi_subgraph.html")
net.show("poi_subgraph.html")

## 8. グラフの保存

In [ ]:
# 結果ディレクトリの作成
results_dir = os.path.join(PROJECT_ROOT, "results")
os.makedirs(results_dir, exist_ok=True)

# グラフをJSON形式で保存
graph_json_path = os.path.join(results_dir, "poi_knowledge_graph.json")
builder.save_graph(graph_json_path, format="json")
print(f"Graph saved to: {graph_json_path}")

# 統計情報を保存
stats_path = os.path.join(results_dir, "graph_stats.json")
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)
print(f"Stats saved to: {stats_path}")

## 9. サマリー

In [ ]:
print("=" * 60)
print("POI Knowledge Graph Construction - Summary")
print("=" * 60)

print(f"""
Input:
  - POI documents: {len(pois):,}

Graph Structure:
  Nodes:
    - POI:         {stats['nodes']['poi']:,}
    - Category:    {stats['nodes']['category']:,}
    - Subcategory: {stats['nodes']['subcategory']:,}
    - Area:        {stats['nodes']['area']:,}
    - Landmark:    {stats['nodes']['landmark']:,}
    - Total:       {stats['nodes']['total']:,}

  Edges:
    - BELONGS_TO:    {stats['edges']['belongs_to']:,}
    - LOCATED_IN:    {stats['edges']['located_in']:,}
    - DISTANCE_FROM: {stats['edges']['distance_from']:,}
    - ADJACENT_TO:   {stats['edges']['adjacent_to']:,}
    - NEAR_TO:       {stats['edges']['near_to']:,}
    - SAME_AREA:     {stats['edges']['same_area']:,}
    - Total:         {stats['edges']['total']:,}

Output Files:
  - {graph_json_path}
  - {stats_path}
  - {os.path.join(results_dir, 'area_structure_graph.png')}
  - {os.path.join(results_dir, 'category_distribution.png')}
  - {os.path.join(results_dir, 'structure_graph.html')}
""")

print("\nNext Steps:")
print("  1. graphrag_02_query_implementation.ipynb - グラフクエリシステムの実装")
print("  2. graphrag_03_evaluation.ipynb - 構造化RAGとの比較評価")